# RSIAT on Google Colab

Chạy code trên SSD tạm của Colab; CIFAR-100, log và checkpoint được giữ trên Google Drive. Trước khi chạy, push các thay đổi local của bạn lên GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

# Set these to the branch that contains your local checkpoint changes.
REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/RSIAT')
OUTPUT_ROOT = Path('/content/drive/MyDrive/RSIAT_runs')
PERSISTENT_DATA_DIR = Path('/content/drive/MyDrive/RSIAT_data/datasets')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_DATA_DIR.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

local_data_dir = PROJECT_DIR / 'data' / 'datasets'
if local_data_dir.is_symlink():
    if local_data_dir.resolve() != PERSISTENT_DATA_DIR.resolve():
        local_data_dir.unlink()
elif local_data_dir.exists():
    if any(local_data_dir.iterdir()):
        shutil.copytree(local_data_dir, PERSISTENT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(local_data_dir)
local_data_dir.parent.mkdir(parents=True, exist_ok=True)
if not local_data_dir.exists():
    local_data_dir.symlink_to(PERSISTENT_DATA_DIR, target_is_directory=True)

os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())
print('Dataset:', PERSISTENT_DATA_DIR)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
%pip install -q -r requirements.txt
!python -m py_compile main.py trainer.py models/base.py models/RSIAT_adapter.py

In [ ]:
# Download CIFAR-100 once to Google Drive.
from torchvision.datasets import CIFAR100
import json

try:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=False)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=False)
    print('CIFAR-100 found in Drive; download skipped.')
except RuntimeError:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=True)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=True)

def make_colab_config(source, destination, seed=1993, resume=False):
    config = json.loads(Path(source).read_text())
    config.update({
        'seed': [seed], 'resume': resume, 'output_root': str(OUTPUT_ROOT),
        'device': ['0'], 'num_workers': 2, 'stats_num_workers': 2,
    })
    Path(destination).write_text(json.dumps(config, indent=2))

make_colab_config('exps/adapter_cifar224_smoke.json', '/content/rsiat_smoke.json')
make_colab_config('exps/adapter_cifar224.json', '/content/rsiat_full.json', resume=True)
print('CIFAR train/test:', len(cifar_train), len(cifar_test))

## Smoke test

Smoke config chạy task đầu tiên với 1 epoch. Nó kiểm tra toàn bộ pipeline RSIAT: data, pretrained ViT, train, evaluation và checkpoint.

In [ ]:
!python main.py --config /content/rsiat_smoke.json

## Full CIFAR-100 experiment

Config full đã bật `resume`: nếu Drive chưa có checkpoint thì chạy từ task 0, ngược lại tự tiếp tục checkpoint task gần nhất. Bỏ comment ô dưới sau khi smoke test thành công.

In [ ]:
# !python main.py --config /content/rsiat_full.json

In [ ]:
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs' / 'adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)